In [1]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, T5ForConditionalGeneration, T5Tokenizer
import torch
import pandas as pd
import json
import gc
import time
pd.set_option('display.max_columns', None) 
pd.set_option('max_colwidth', None) # show full width of showing cols
pd.set_option("expand_frame_repr", False) # print cols side by side as it's supposed to be
from utils.config import q_type_models, qa_type_models

In [2]:
with open("Model_dataset/cv.json", "r") as file:
    CV_DATA= json.load(file)

In [3]:
questions = [
    "What have you used for web development?",
    "How do I train a neural network?",
    "How much do you want to earn?",
    "What are you looking for in the new company?",
    "How much break do you need per day?",
    "Share you most recent salary draw.",
    "Rate yourself 1 to 10 for python developer.",
    "What is your most recent degree?",
    "Have you completed masters?",
    "Have you completed Bsc",
    "Who is the CEO of your company?",
    "Who inspire you the most?",
    "have you rechived any performance bounus?",
    "How frequent you expect performance review?",
    "Have you review others code before?"
    "What are the top framework you use?",
    "Have you completed any personal project recently?",
    "Have you a lead a team before?",
    "How many years of experience do you have with SQL?",
    "What you expect from the current company?",
    "How many years of work experience do you have with Microsoft Products?",
    "How many years of work experience do you have with Microsoft Fabric?",
    "Have you completed the following level of education: Bachelor's Degree?",
    "Mobile phone number",
    "Phone country code",
    "Email address",
    "How many years of work experience do you have with Python (Programming Language)?",
    "How many years of work experience do you have with Google BigQuery?",
    "How many years of work experience do you have with Terraform?",
    "What have you used for web development?",
    "How do I train a neural network?",
    "How much do you want to earn?",
    "What are you looking for in the new company?",
    "How much break do you need per day?",
    "Share you most recent salary draw.",
    "Rate yourself 1 to 10 for python developer.",
    "What is your most recent degree?",
    "Have you completed masters?",
    "Have you completed Bsc",
    "Who is the CEO of your company?",
    "Who inspire you the most?",
    "have you rechived any performance bounus?",
    "How frequent you expect performance review?",
    "Have you review others code before?"
    "What are the top framework you use?",
    "Have you completed any personal project recently?",
    "Have you a lead a team before?",
    "How many years of experience do you have with SQL?",
    "What you expect from the current company?",
    "How many years of work experience do you have with Microsoft Products?",
    "How many years of work experience do you have with Microsoft Fabric?",
    "Have you completed the following level of education: Bachelor's Degree?",
    "Mobile phone number",
    "Phone country code",
    "Email address",
    "How many years of work experience do you have with Python (Programming Language)?",
    "How many years of work experience do you have with Google BigQuery?",
    "How many years of work experience do you have with Terraform?",
]

# Using classifier

### For single input at a time

In [4]:
def get_question_type_prediction(text):
    # Load model & tokenizer inside function (so it's released later)
    QUESTION_CLASSIFIER_MODEL = DistilBertForSequenceClassification.from_pretrained(q_type_models[-1])
    QUESTION_CLASSIFIER_TOKENIZER = DistilBertTokenizerFast.from_pretrained(q_type_models[-1])
    ID2LABEL = QUESTION_CLASSIFIER_MODEL.config.id2label
    try:
        torch.cuda.empty_cache()
        gc.collect()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        QUESTION_CLASSIFIER_MODEL.to(device)
        QUESTION_CLASSIFIER_MODEL.eval()
        inputs = QUESTION_CLASSIFIER_TOKENIZER(text, padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = QUESTION_CLASSIFIER_MODEL(**inputs)
        logits = outputs.logits
        predicted_classes = torch.argmax(logits, dim=1)
        result = ID2LABEL[predicted_classes.item()]
    except torch.cuda.OutOfMemoryError:
        print("CUDA Out of Memory! Consider using a smaller batch size or switching to CPU.")
    finally:
        # Cleanup: Release memory after execution
        del QUESTION_CLASSIFIER_MODEL, QUESTION_CLASSIFIER_TOKENIZER
        torch.cuda.empty_cache()
        gc.collect()
    return result
get_question_type_prediction(questions[-1])

'skills'

### For multiple input at a time/ batch processing

In [5]:
def get_question_type_predictions(texts, batch_size= 64):
    # Load model & tokenizer inside function (so it's released later)
    QUESTION_CLASSIFIER_MODEL = DistilBertForSequenceClassification.from_pretrained(q_type_models[-1])
    QUESTION_CLASSIFIER_TOKENIZER = DistilBertTokenizerFast.from_pretrained(q_type_models[-1])
    ID2LABEL = QUESTION_CLASSIFIER_MODEL.config.id2label
    results = []
    try:
        torch.cuda.empty_cache()
        gc.collect()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        QUESTION_CLASSIFIER_MODEL.to(device)
        QUESTION_CLASSIFIER_MODEL.eval()
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            inputs = QUESTION_CLASSIFIER_TOKENIZER(batch_texts, padding=True, truncation=True, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = QUESTION_CLASSIFIER_MODEL(**inputs)
            logits = outputs.logits
            predicted_classes = torch.argmax(logits, dim=1)
            batch_results = [ID2LABEL[idx.item()] for idx in predicted_classes]
            results.extend(batch_results)
            del inputs, outputs, logits
            torch.cuda.empty_cache()
            gc.collect()
    except torch.cuda.OutOfMemoryError:
        print("CUDA Out of Memory! Consider reducing batch size further or using CPU.")
    except Exception as e:
        print(f"Error occurred: {e}")
    finally:
        del QUESTION_CLASSIFIER_MODEL, QUESTION_CLASSIFIER_TOKENIZER
        torch.cuda.empty_cache()
        gc.collect()
    return results

# Example usage
get_question_type_predictions(questions)

['working_experience',
 'working_experience',
 'expected_ctc',
 'expected_ctc',
 'expected_ctc',
 'current_ctc',
 'skills',
 'education',
 'working_experience',
 'education',
 'working_experience',
 'others',
 'working_experience',
 'expected_ctc',
 'working_experience',
 'working_experience',
 'working_experience',
 'skills',
 'expected_ctc',
 'skills',
 'skills',
 'education',
 'personal_information',
 'personal_information',
 'personal_information',
 'skills',
 'skills',
 'skills',
 'working_experience',
 'working_experience',
 'expected_ctc',
 'expected_ctc',
 'expected_ctc',
 'current_ctc',
 'skills',
 'education',
 'working_experience',
 'education',
 'working_experience',
 'others',
 'working_experience',
 'expected_ctc',
 'working_experience',
 'working_experience',
 'working_experience',
 'skills',
 'expected_ctc',
 'skills',
 'skills',
 'education',
 'personal_information',
 'personal_information',
 'personal_information',
 'skills',
 'skills',
 'skills']

### Comparing results for all different optimiser result:
- multiple input for all the avilable classifier

In [6]:
def get_dataframe_for_comparision_question_classifier(questions, batch_size=64):
    table_data = {"questions": questions}
    for q_type_model in q_type_models:
        # Load model & tokenizer inside loop (so they are released after each iteration)
        QUESTION_CLASSIFIER_MODEL = DistilBertForSequenceClassification.from_pretrained(q_type_model)
        QUESTION_CLASSIFIER_TOKENIZER = DistilBertTokenizerFast.from_pretrained(q_type_model)
        ID2LABEL = QUESTION_CLASSIFIER_MODEL.config.id2label
        predictions = []
        try:
            torch.cuda.empty_cache()
            gc.collect()
            device = "cuda" if torch.cuda.is_available() else "cpu"
            QUESTION_CLASSIFIER_MODEL.to(device)
            QUESTION_CLASSIFIER_MODEL.eval()
            for i in range(0, len(questions), batch_size):
                batch_questions = questions[i:i + batch_size]
                inputs = QUESTION_CLASSIFIER_TOKENIZER(batch_questions, padding=True, truncation=True, return_tensors="pt").to(device)
                with torch.no_grad():
                    outputs = QUESTION_CLASSIFIER_MODEL(**inputs)
                logits = outputs.logits
                predicted_classes = torch.argmax(logits, dim=1)
                batch_predictions = [ID2LABEL[idx.item()] for idx in predicted_classes]
                predictions.extend(batch_predictions)
                del inputs, outputs, logits
                torch.cuda.empty_cache()
                gc.collect()
        except torch.cuda.OutOfMemoryError:
            print(f"CUDA Out of Memory for model {q_type_model}! Consider reducing batch size further or switching to CPU.")
            predictions = None
        except Exception as e:
            print(f"Error occurred with model {q_type_model}: {e}")
            predictions = None
        finally:
            del QUESTION_CLASSIFIER_MODEL, QUESTION_CLASSIFIER_TOKENIZER
            torch.cuda.empty_cache()
            gc.collect()
        table_data[q_type_model.split("-")[-1]] = predictions
    return pd.DataFrame(table_data)


# Example usage
get_dataframe_for_comparision_question_classifier(questions)

,questions,default,Adam,AdamW,SGD,v2
0,What have you used for web development?,working_experience,working_experience,working_experience,working_experience,working_experience
1,How do I train a neural network?,working_experience,working_experience,working_experience,working_experience,working_experience
2,How much do you want to earn?,expected_ctc,expected_ctc,expected_ctc,expected_ctc,expected_ctc
3,What are you looking for in the new company?,availability,availability,personal_information,expected_ctc,expected_ctc
4,How much break do you need per day?,expected_ctc,expected_ctc,availability,expected_ctc,expected_ctc
5,Share you most recent salary draw.,current_ctc,current_ctc,current_ctc,current_ctc,current_ctc
6,Rate yourself 1 to 10 for python developer.,skills,skills,skills,skills,skills
7,What is your most recent degree?,education,education,education,education,education
8,Have you completed masters?,education,education,education,working_experience,working_experience
9,Have you completed Bsc,education,education,education,education,education


# Question answer model

In [7]:
qa_type_models= [
    ".temp/model_results/fine_tuned_question_answer_model-base/checkpoint-2960",
    ".temp/model_results/fine_tuned_question_answer_model-base/checkpoint-4440",
    ".temp/model_results/fine_tuned_question_answer_model-base-temp/checkpoint-740",
    "model/fine_tuned_question_answer_model-base",
    ]

### For single input at a time

In [8]:
def get_qa_model_out_raw(question):
    # Load model & tokenizer inside function (so they are released after execution)
    QUESTION_ANSWER_MODEL = T5ForConditionalGeneration.from_pretrained(qa_type_models[-1])
    QUESTION_ANSWER_TOKENIZER = T5Tokenizer.from_pretrained(qa_type_models[-1], legacy=False)
    try:
        torch.cuda.empty_cache()
        gc.collect()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        QUESTION_ANSWER_MODEL.to(device)
        QUESTION_ANSWER_MODEL.eval()
        # Get predicted question type
        predicted_question_type = get_question_type_prediction(question)
        context = CV_DATA.get(predicted_question_type, "")
        input_text = f"question: {question} context: {context}"
        inputs = QUESTION_ANSWER_TOKENIZER(input_text, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = QUESTION_ANSWER_MODEL.generate(
                input_ids=inputs["input_ids"], max_length=50, num_beams=4, early_stopping=True
            )
        result = QUESTION_ANSWER_TOKENIZER.decode(outputs[0], skip_special_tokens=True)
    except torch.cuda.OutOfMemoryError:
        print("CUDA Out of Memory! Consider reducing input size or using CPU.")
        result = None
    finally:
        # Cleanup: Release memory after execution
        del QUESTION_ANSWER_MODEL, QUESTION_ANSWER_TOKENIZER
        torch.cuda.empty_cache()
        gc.collect()
    return predicted_question_type, result


# Example usage
get_qa_model_out_raw(questions[-1])

('skills', '2')

### For multiple input at a time/ batch processing

In [11]:
"""
    For my configuration nvidia 1650Ti 4GB graphic batch size=4 is the max my system can procede
"""
def get_qa_model_out_batch(questions, batch_size=4):
    QUESTION_ANSWER_MODEL = T5ForConditionalGeneration.from_pretrained(qa_type_models[-1])
    QUESTION_ANSWER_TOKENIZER = T5Tokenizer.from_pretrained(qa_type_models[-1], legacy=False)
    inputs = None
    decoded_outputs = []
    predicted_types = []
    try:
        torch.cuda.empty_cache()
        gc.collect()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        QUESTION_ANSWER_MODEL.to(device)
        QUESTION_ANSWER_MODEL.eval()
        for i in range(0, len(questions), batch_size):
            batch_questions = questions[i:i + batch_size]
            batch_predicted_types = [get_question_type_prediction(q) for q in batch_questions]
            batch_contexts = [CV_DATA.get(p_type, "") for p_type in batch_predicted_types]
            batch_input_texts = [f"question: {q} context: {c}" for q, c in zip(batch_questions, batch_contexts)]
            inputs = QUESTION_ANSWER_TOKENIZER(batch_input_texts, padding=True, truncation=True, return_tensors="pt").to(device)
            with torch.no_grad():
                batch_outputs = QUESTION_ANSWER_MODEL.generate(input_ids=inputs["input_ids"], max_length=50, num_beams=4, early_stopping=True)
            batch_decoded_outputs = [QUESTION_ANSWER_TOKENIZER.decode(out, skip_special_tokens=True) for out in batch_outputs]
            decoded_outputs.extend(batch_decoded_outputs)
            predicted_types.extend(batch_predicted_types)
            del inputs, batch_outputs, batch_input_texts
            torch.cuda.empty_cache()
            gc.collect()
        if not decoded_outputs:
            decoded_outputs = [""] * len(questions)
    except torch.cuda.OutOfMemoryError:
        print("CUDA Out of Memory! Consider reducing batch size further or using CPU.")
        decoded_outputs = [""] * len(questions)
    except Exception as e:
        print(f"Error occurred: {e}")
        decoded_outputs = [""] * len(questions)
    finally:
        del QUESTION_ANSWER_MODEL, QUESTION_ANSWER_TOKENIZER
        torch.cuda.empty_cache()
        gc.collect()

    # Return results as a DataFrame
    result_df = pd.DataFrame({
        'Question': questions,
        'Question Type': predicted_types,
        'Answer': decoded_outputs
    })
    return result_df

# Example usage
get_qa_model_out_batch(questions)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


,Question,Question Type,Answer
0,What have you used for web development?,working_experience,"Apache Airflow, dbt, Snowflake, Redis."
1,How do I train a neural network?,working_experience,Using neural networks to train neural networks.
2,How much do you want to earn?,expected_ctc,850000
3,What are you looking for in the new company?,expected_ctc,Varies
4,How much break do you need per day?,expected_ctc,Varies
5,Share you most recent salary draw.,current_ctc,Varies
6,Rate yourself 1 to 10 for python developer.,skills,8
7,What is your most recent degree?,education,Master of Computer Application (MCA)
8,Have you completed masters?,working_experience,No.
9,Have you completed Bsc,education,Master's degree


### Comparing results for all different optimiser result:
- multiple input for all the avilable qa models

In [10]:
def get_dataframe_for_comparision_question_answer(questions, batch_size=4):
    data= {"questions": questions}
    model_index= 0
    for qa_type_model in qa_type_models:
        model_index+= 1
        QUESTION_ANSWER_MODEL = T5ForConditionalGeneration.from_pretrained(qa_type_model)
        QUESTION_ANSWER_TOKENIZER = T5Tokenizer.from_pretrained(qa_type_models[-1], legacy=False)
        inputs = None
        decoded_outputs = []
        predicted_types = []
        try:
            torch.cuda.empty_cache()
            gc.collect()
            device = "cuda" if torch.cuda.is_available() else "cpu"
            QUESTION_ANSWER_MODEL.to(device)
            QUESTION_ANSWER_MODEL.eval()
            for i in range(0, len(questions), batch_size):
                batch_questions = questions[i:i + batch_size]
                batch_predicted_types = [get_question_type_prediction(q) for q in batch_questions]
                batch_contexts = [CV_DATA.get(p_type, "") for p_type in batch_predicted_types]
                batch_input_texts = [f"question: {q} context: {c}" for q, c in zip(batch_questions, batch_contexts)]
                inputs = QUESTION_ANSWER_TOKENIZER(batch_input_texts, padding=True, truncation=True, return_tensors="pt").to(device)
                with torch.no_grad():
                    batch_outputs = QUESTION_ANSWER_MODEL.generate(input_ids=inputs["input_ids"], max_length=50, num_beams=4, early_stopping=True)
                batch_decoded_outputs = [QUESTION_ANSWER_TOKENIZER.decode(out, skip_special_tokens=True) for out in batch_outputs]
                decoded_outputs.extend(batch_decoded_outputs)
                predicted_types.extend(batch_predicted_types)
                del inputs, batch_outputs, batch_input_texts
                torch.cuda.empty_cache()
                gc.collect()
            if not decoded_outputs:
                decoded_outputs = [""] * len(questions)
        except torch.cuda.OutOfMemoryError:
            print("CUDA Out of Memory! Consider reducing batch size further or using CPU.")
            decoded_outputs = [""] * len(questions)
        except Exception as e:
            print(f"Error occurred: {e}")
            decoded_outputs = [""] * len(questions)
        finally:
            del QUESTION_ANSWER_MODEL, QUESTION_ANSWER_TOKENIZER
            torch.cuda.empty_cache()
            gc.collect()
        data[f"model_out_{data}"]= decoded_outputs
    return pd.DataFrame(data)



get_dataframe_for_comparision_question_answer(questions)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


KeyboardInterrupt: 